In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 01:24:27.127136: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 01:24:27.795948: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Rain is initialized
Provisioner: Creating coordinator
Coordinator initialized successfully
LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_sync()

Rain: Creating workers
provisioner is serving
Provisioner: Starting coordinator
coordinator is serving
Coordinator: sending the num of workers to the provisioner
Number of workers is: 3
divider received: Success receiving the number of workers from provisioner
LocalProvisioner: Creating workers
[Created workers]
 IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
Worker is running on port: 50151
Worker is running on port: 50152
Transceiver is serving
Rain: Sending data to workers
divider is sending data to the coordinator
Worker is running on port: 50153
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinato

2023-07-02 01:24:57.219587: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 01:24:57.222990: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 01:24:57.243128: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 10ms/step - loss: 0.7047 - accuracy: 0.7781
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.7017 - accuracy: 0.7776
Epoch 2/2
157/157 [==============================] - 2s 11ms/step - loss: 0.6910 - accuracy: 0.7836
Epoch 2/2
157/157 [==============================] - 1s 8ms/step - loss: 0.3102 - accuracy: 0.9068
sending data to coordinator
sending data to coordinator
157/157 [==============================] - 1s 7ms/step - loss: 0.3093 - accuracy: 0.9066
sending data to coordinator
coordinator received: Executed! from worker 
coordinator received: Executed! from worker 
thread 1 is done
coordinator received: Executed! from worker 
Downloaded ../../../Coordinator/coord/data/1_1_trained.pkl in coordinator
thread 2 is done
Downloaded ../../../Coordinator/coord/data/2_1_trained.pkl in coordinator
thread 3 is done
Downloaded ../../../Coordinator/coord/data/3_1_trained.pkl in coordinator


2023-07-02 01:25:19.459310: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 01:25:19.460751: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 01:25:19.464434: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 9ms/step - loss: 0.2630 - accuracy: 0.9225
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.2615 - accuracy: 0.9209
Epoch 2/2
157/157 [==============================] - 2s 10ms/step - loss: 0.2028 - accuracy: 0.9374
sending data to coordinator
157/157 [==============================] - 1s 9ms/step - loss: 0.1978 - accuracy: 0.9397
sending data to coordinator
coordinator received: Executed! from worker 
thread 1 is done
coordinator received: Executed! from worker 
Downloaded ../../../Coordinator/coord/data/1_2_trained.pkl in coordinator
thread 2 is done
Downloaded ../../../Coordinator/coord/data/2_2_trained.pkl in coordinator
sending data to coordinator
coordinator received: Executed! from worker 
thread 3 is done
Downloaded ../../../Coordinator/coord/data/3_2_trained.pkl in coordinator
coordinator received: Success! from divider 
coordinator received: Success! from divider 
coordinato

2023-07-02 01:25:42.954034: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 01:25:42.955350: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 01:25:42.957536: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 9ms/step - loss: 0.1801 - accuracy: 0.9459
Epoch 2/2
157/157 [==============================] - 2s 9ms/step - loss: 0.1826 - accuracy: 0.9445
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.1570 - accuracy: 0.9539
sending data to coordinator
157/157 [==============================] - 1s 9ms/step - loss: 0.1540 - accuracy: 0.9546
sending data to coordinator
coordinator received: Executed! from worker 
thread 1 is done
coordinator received: Executed! from worker 
coordinator received: Executed! from worker 
Downloaded ../../../Coordinator/coord/data/1_3_trained.pkl in coordinator
thread 2 is done
Downloaded ../../../Coordinator/coord/data/2_3_trained.pkl in coordinator
thread 3 is done
Downloaded ../../../Coordinator/coord/data/3_3_trained.pkl in coordinator
coordinator received: Success! from divider 
coordinator received: Success! from divider 
coordinator received: Success! from divi

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0998 - accuracy: 0.9696

Test accuracy: 97.0%
